In [ ]:
# 1. Install; restart the session, then continue with cell 2.
!nvidia-smi
!git clone --branch experiment/custom-steering-vectors --single-branch https://github.com/Nicerova7/show-o2-space.git /content/show-o2-space
!git clone https://github.com/showlab/Show-o.git /content/Show-o
!git -C /content/Show-o checkout 45a5a2de01d1ebd10cd5864d29310a76476cdf23
%pip install -q transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1

In [ ]:
# 2. Load frozen Show-o2.
import importlib.metadata
import json
import os
import sys
from pathlib import Path

import torch
from huggingface_hub import model_info, notebook_login

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, UPSTREAM_REVISION, ShowO2Adapter
from experiments.custom_utils import (
    save_json, capture_pair, reduce_states,
    save_table, show_example, summarize, steer_understanding,
)

SEED = 42
SYSTEM = "You are a helpful assistant."
torch.manual_seed(SEED)
adapter = ShowO2Adapter()
OUTPUT = PROJECT / "outputs/safety"
OUTPUT.mkdir(parents=True, exist_ok=True)
BUILD = OUTPUT / "vectors"
VECTOR_FILE = BUILD / "vectors.pt"
generation = dict(seed=SEED, max_new_tokens=128)
runtime = dict(
    model=MODEL_ID, model_revision=model_info(MODEL_ID).sha,
    upstream_revision=UPSTREAM_REVISION, dtype=str(adapter.dtype),
    tokenizer=adapter.tokenizer.name_or_path, chat_template=adapter.tokenizer.chat_template,
    system=SYSTEM, use_cache=False,
    packages={name: importlib.metadata.version(name) for name in
              ("torch", "torchvision", "transformers", "diffusers", "huggingface-hub")},
)
print(MODEL_ID, "|", adapter.dtype, "| Frozen:", not any(p.requires_grad for p in adapter.model.parameters()))


In [ ]:
# 3. Sample Refusal training instructions; no image is used.
import random
from urllib.request import urlopen

REFUSAL_REVISION = "9d852fae1a9121c78b29142de733cb1340770cc3"
training_path = OUTPUT / "training.json"

if training_path.exists():
    training = json.loads(training_path.read_text(encoding="utf-8"))
else:
    groups = {}
    for group in ("harmful", "harmless"):
        url = (f"https://raw.githubusercontent.com/andyrdt/refusal_direction/{REFUSAL_REVISION}"
               f"/dataset/splits/{group}_train.json")
        with urlopen(url) as response:
            records = json.load(response)
        indices = random.Random(SEED).sample(range(len(records)), 128)
        groups[group] = [dict(source_index=i, **records[i]) for i in indices]
    training = dict(revision=REFUSAL_REVISION, seed=SEED, groups=groups)
    save_json(training_path, training)

print("Refusal revision:", training["revision"])
print({group: len(rows) for group, rows in training["groups"].items()})


In [ ]:
# Inspect one independent example from each training group.
print("HARMFUL:\n", training["groups"]["harmful"][0]["instruction"])
print("\nHARMLESS:\n", training["groups"]["harmless"][0]["instruction"])


In [ ]:
# 4. Format text-only instructions and inspect the final chat tokens.
formatted = {group: [] for group in ("harmful", "harmless")}
audit = []

for group, instructions in training["groups"].items():
    for example in instructions:
        messages = [dict(role="system", content=SYSTEM),
                    dict(role="user", content=example["instruction"])]
        ids = adapter.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
        )
        formatted[group].append(ids)
        audit.append(dict(group=group, source_index=example["source_index"], token_ids=ids,
                          position=len(ids) - 1, token=adapter.tokenizer.convert_ids_to_tokens(ids[-1])))

for group, sequences in formatted.items():
    print(group, "| Last tokens:", adapter.tokenizer.convert_ids_to_tokens(sequences[0][-6:]))


In [ ]:
# 5. Capture the last post-instruction token at every block output; no image.
layers = list(range(len(adapter.layers)))
harmful = {layer: [] for layer in layers}
harmless = {layer: [] for layer in layers}

for index, (positive_ids, negative_ids) in enumerate(zip(formatted["harmful"], formatted["harmless"])):
    pair = dict(token_ids=dict(positive=positive_ids, negative=negative_ids))
    captured = capture_pair(adapter, pair, layers=layers)
    for layer in layers:
        harmful[layer].append(captured["positive"][layer][-1].clone())
        harmless[layer].append(captured["negative"][layer][-1].clone())
    print(index + 1, "/", len(formatted["harmful"]))

harmful = {layer: torch.stack(states) for layer, states in harmful.items()}
harmless = {layer: torch.stack(states) for layer, states in harmless.items()}
print("Layer:", layers[0], "| Harmful:", harmful[layers[0]].shape, "| Harmless:", harmless[layers[0]].shape)


In [ ]:
# 6. Mean(harmful) - mean(harmless), then unit normalization per layer.
means = {
    "harmful": {layer: reduce_states(states, "mean") for layer, states in harmful.items()},
    "harmless": {layer: reduce_states(states, "mean") for layer, states in harmless.items()},
}
directions = {layer: means["harmful"][layer] - means["harmless"][layer] for layer in layers}
vectors = {layer: vector / vector.norm() for layer, vector in directions.items()}

for layer, vector in vectors.items():
    print("Layer:", layer, "| Shape:", tuple(vector.shape), "| Norm:", vector.norm().item())


In [ ]:
# 7. Save the vectors, training IDs, and token audit.
BUILD.mkdir(parents=True, exist_ok=True)
metadata = dict(
    runtime=runtime, seed=SEED, source_revision=training["revision"],
    training_ids={group: [row["source_index"] for row in records]
                  for group, records in training["groups"].items()},
    n_harmful=len(formatted["harmful"]), n_harmless=len(formatted["harmless"]),
    modality="text-only; no image", capture="decoder block OUTPUT before final norm",
    position="last chat-template token after instruction, with assistant generation prefix",
    direction="mean(harmful) - mean(harmless)", normalization="unit per layer",
    layers=layers, paired=False,
)
torch.save(dict(vectors=vectors, means=means, metadata=metadata), VECTOR_FILE)
save_json(BUILD / "metadata.json", metadata)
save_json(BUILD / "token_audit.json", audit)
save_json(BUILD / "training.json", training)
print("Saved:", VECTOR_FILE)


In [ ]:
# 8. Reload saved vectors; on later sessions, run cell 2 and jump here.
saved = torch.load(VECTOR_FILE, map_location="cpu", weights_only=True)
vectors = saved["vectors"]
print("Loaded:", VECTOR_FILE, "| Layers:", list(vectors))


In [ ]:
# 9. Authenticate and load VLGuard metadata.
from huggingface_hub import hf_hub_download

VLGUARD_REVISION = "b0be37a1ab7accb14e10d6a0ec3ce62cfaff2d46"
# Accept dataset access on Hugging Face before running this cell.
notebook_login()
vlguard_metadata = {}

for source_split in ("train", "test"):
    metadata_file = hf_hub_download(
        "ys-zong/VLGuard", f"{source_split}.json",
        repo_type="dataset", revision=VLGUARD_REVISION,
    )
    vlguard_metadata[source_split] = json.loads(Path(metadata_file).read_text(encoding="utf-8"))
    print(source_split, "| Records:", len(vlguard_metadata[source_split]))


In [ ]:
# 10. Select 32 safe train images and 100 safe test images; reuse saved IDs.
import random

samples_path = OUTPUT / "samples.json"
if samples_path.exists():
    samples = json.loads(samples_path.read_text(encoding="utf-8"))
else:
    selected_splits = {}
    for split, source_split, count in (("validation", "train", 32), ("test", "test", 100)):
        safe = [(i, row) for i, row in enumerate(vlguard_metadata[source_split]) if row["safe"] is True]
        safe.sort(key=lambda item: item[1]["image"])
        selected = random.Random(SEED).sample(safe, count)
        selected_splits[split] = []
        for index, row in selected:
            instructions = row["instr-resp"]
            selected_splits[split].append(dict(
                id=row["image"], source_index=index, source_split=source_split,
                image=str(Path("images") / source_split / Path(row["image"]).name),
                harmful=next(item["unsafe_instruction"] for item in instructions if "unsafe_instruction" in item),
                harmless=next(item["safe_instruction"] for item in instructions if "safe_instruction" in item),
            ))
    samples = dict(revision=VLGUARD_REVISION, seed=SEED, splits=selected_splits)
    save_json(samples_path, samples)

print({split: len(rows) for split, rows in samples["splits"].items()})
print("Fixed IDs:", samples_path)


In [ ]:
# 11. Download the image archives and extract only the selected images.
import shutil
from zipfile import ZipFile

for split, examples in samples["splits"].items():
    source_split = examples[0]["source_split"]
    archive = hf_hub_download(
        "ys-zong/VLGuard", f"{source_split}.zip",
        repo_type="dataset", revision=samples["revision"],
    )
    (OUTPUT / "images" / source_split).mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as zipped:
        members = {Path(name).name: name for name in zipped.namelist() if not name.endswith("/")}
        for example in examples:
            destination = OUTPUT / example["image"]
            with zipped.open(members[destination.name]) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
    print(split, "| Images ready:", len(examples))


In [ ]:
# 12. Select baseline examples and load saved responses.
baseline_split = "validation"
baseline_example_limit = 4  # Images; each has a harmful and a harmless instruction.

BASELINES = OUTPUT / "baselines"
BASELINES.mkdir(parents=True, exist_ok=True)
save_json(BASELINES / "metadata.json", dict(runtime=runtime, generation=generation))
baseline_file = BASELINES / "responses.json"
baseline_rows = json.loads(baseline_file.read_text(encoding="utf-8")) if baseline_file.exists() else []
baseline_lookup = {(row["split"], row["sample_id"], row["instruction_type"]): row for row in baseline_rows}
baseline_examples = samples["splits"][baseline_split][:baseline_example_limit]
print("Images:", len(baseline_examples), "| Instructions:", 2 * len(baseline_examples))


In [ ]:
# 12.1. Generate missing baselines without steering; save each response.
for example in baseline_examples:
    for group in ("harmful", "harmless"):
        key = (baseline_split, example["id"], group)
        if key not in baseline_lookup:
            torch.manual_seed(generation["seed"])
            embeddings, mask, _ = adapter.prepare_understanding(OUTPUT / example["image"], example[group])
            answer = adapter.answer(embeddings, mask, max_new_tokens=generation["max_new_tokens"])
            row = dict(
                split=baseline_split, sample_id=example["id"], source_split=example["source_split"],
                image=example["image"], instruction_type=group, instruction=example[group],
                baseline=answer, baseline_refusal="", baseline_incoherent="", baseline_irrelevant="",
            )
            baseline_rows.append(row)
            baseline_lookup[key] = row
            save_table(baseline_file, baseline_rows)
        print(example["id"], "|", group, "| ready")
print("Saved:", baseline_file)


In [ ]:
# 12.2. Review each response, choose manual labels, and click Save labels.
import ipywidgets as widgets
from IPython.display import display

annotation_rows = json.loads(baseline_file.read_text(encoding="utf-8"))
annotation_examples = {example["id"]: example for example in baseline_examples}
review_rows = [row for row in annotation_rows
               if row["split"] == baseline_split and row["sample_id"] in annotation_examples]

for row in review_rows:
    print("Sample:", row["sample_id"], "|", row["instruction_type"])
    show_example(OUTPUT, annotation_examples[row["sample_id"]], row["instruction"], row["baseline"])
    controls = {
        field: widgets.ToggleButtons(
            options=[("Unlabelled", ""), ("No (0)", 0), ("Yes (1)", 1)],
            value=row["baseline_" + field], description=field.capitalize(),
            style={"description_width": "initial"},
        ) for field in ("refusal", "incoherent", "irrelevant")
    }
    button = widgets.Button(description="Save labels")

    def save_labels(button, row=row, controls=controls, rows=annotation_rows, path=baseline_file):
        for field, control in controls.items():
            row["baseline_" + field] = control.value
        save_table(path, rows)
        button.description = "Saved"
        button.button_style = "success"

    button.on_click(save_labels)
    display(widgets.VBox([*controls.values(), button]))


In [ ]:
# 12.3. Summarize the labels saved for these examples.
labelled_rows = json.loads(baseline_file.read_text(encoding="utf-8"))
selected_baselines = [row for row in labelled_rows
                      if row["split"] == baseline_split and row["sample_id"] in annotation_examples]
summarize(selected_baselines)
print("Labels:", baseline_file, "| CSV:", baseline_file.with_suffix(".csv"))


In [ ]:
# 13. Selected-layer configuration; limits count images.
intervention_layers = [7]  # Editable to [7, 8, 12], etc.
target = "text"           # "text", "visual", "text+visual"
operation = "add"         # "add", "component"
coefficient = 1.0
split = "validation"
example_limit = 4  # Images; validation allows 32 harmful + 32 harmless prompts.

selected_configuration = dict(
    intervention_layers=intervention_layers, target=target, operation=operation,
    coefficient=coefficient, split=split, example_limit=example_limit,
)


In [ ]:
# 14. Select examples and snapshot this run's configuration.
from datetime import datetime, timezone

selected_pool = samples["splits"][selected_configuration["split"]]
if selected_configuration["example_limit"] > len(selected_pool):
    raise ValueError(
        f"{selected_configuration['split']} has {len(selected_pool)} images "
        f"({len(selected_pool)} harmful + {len(selected_pool)} harmless prompts); "
        f"requested {selected_configuration['example_limit']} images."
    )
selected_examples = selected_pool[:selected_configuration["example_limit"]]
baseline_rows = json.loads((BASELINES / "responses.json").read_text(encoding="utf-8"))
baseline_lookup = {(row["split"], row["sample_id"], row["instruction_type"]): row for row in baseline_rows}
selected_inputs = [
    (example, group, baseline_lookup[(selected_configuration["split"], example["id"], group)])
    for example in selected_examples for group in ("harmful", "harmless")
]
selected_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
selected_run = OUTPUT / "runs" / "selected_layers" / selected_run_id
selected_run_config = dict(
    selected_configuration, generation=dict(generation), runtime=runtime, vector_file=str(VECTOR_FILE),
    baseline_directory=str(BASELINES), sample_ids=[row["id"] for row in selected_examples],
    vlguard_revision=samples["revision"], decoding="greedy, full-prefix, use_cache=False",
)
print("Configuration:", selected_configuration)


In [ ]:
# 15. Apply steering using the saved vectors and baselines.
selected_rows, selected_traces = [], []

for example, group, baseline in selected_inputs:
    torch.manual_seed(selected_run_config["generation"]["seed"])
    embeddings, mask, alignment = adapter.prepare_understanding(OUTPUT / example["image"], example[group])
    with steer_understanding(
        adapter, vectors=vectors, alignment=alignment, layers=selected_run_config["intervention_layers"],
        target=selected_run_config["target"], operation=selected_run_config["operation"],
        coefficient=selected_run_config["coefficient"],
    ) as trace:
        answer = adapter.answer(embeddings, mask, max_new_tokens=selected_run_config["generation"]["max_new_tokens"])
    selected_rows.append(dict(
        baseline, steered=answer, steered_refusal="", steered_incoherent="", steered_irrelevant="",
        run_id=selected_run_id, target=selected_run_config["target"], operation=selected_run_config["operation"],
        coefficient=selected_run_config["coefficient"],
        intervention_layers=json.dumps(selected_run_config["intervention_layers"]),
    ))
    selected_traces.append(dict(sample_id=example["id"], instruction_type=group, alignment=alignment, **trace))
    print(example["id"], "|", group, "| completed")


In [ ]:
# 16. Compare baseline and steered responses.
for (example, group, _), row in zip(selected_inputs, selected_rows):
    show_example(OUTPUT, example, row["instruction"], row["baseline"], row["steered"])


In [ ]:
# 17. Save this run independently; refusal labels remain empty.
selected_run.mkdir(parents=True, exist_ok=True)
save_json(selected_run / "configuration.json", selected_run_config)
save_table(selected_run / "responses.json", selected_rows)
save_json(selected_run / "trace.json", selected_traces)
print("Saved:", selected_run)


In [ ]:
# 18. Independent all-layer configuration; limits count images.
intervention_layers = "all"
target = "text"           # "text", "visual", "text+visual"
operation = "add"         # "add", "component"
coefficient = 1.0
split = "validation"
example_limit = 4  # Images; validation allows 32 harmful + 32 harmless prompts.

all_configuration = dict(
    intervention_layers=intervention_layers, target=target, operation=operation,
    coefficient=coefficient, split=split, example_limit=example_limit,
)


In [ ]:
# 19. Select examples and snapshot this run's configuration.
from datetime import datetime, timezone

all_pool = samples["splits"][all_configuration["split"]]
if all_configuration["example_limit"] > len(all_pool):
    raise ValueError(
        f"{all_configuration['split']} has {len(all_pool)} images "
        f"({len(all_pool)} harmful + {len(all_pool)} harmless prompts); "
        f"requested {all_configuration['example_limit']} images."
    )
all_examples = all_pool[:all_configuration["example_limit"]]
baseline_rows = json.loads((BASELINES / "responses.json").read_text(encoding="utf-8"))
baseline_lookup = {(row["split"], row["sample_id"], row["instruction_type"]): row for row in baseline_rows}
all_inputs = [
    (example, group, baseline_lookup[(all_configuration["split"], example["id"], group)])
    for example in all_examples for group in ("harmful", "harmless")
]
all_run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
all_run = OUTPUT / "runs" / "all_layers" / all_run_id
all_run_config = dict(
    all_configuration, generation=dict(generation), runtime=runtime, vector_file=str(VECTOR_FILE),
    baseline_directory=str(BASELINES), sample_ids=[row["id"] for row in all_examples],
    vlguard_revision=samples["revision"], decoding="greedy, full-prefix, use_cache=False",
)
print("Configuration:", all_configuration)


In [ ]:
# 20. Apply steering using the saved vectors and baselines.
all_rows, all_traces = [], []

for example, group, baseline in all_inputs:
    torch.manual_seed(all_run_config["generation"]["seed"])
    embeddings, mask, alignment = adapter.prepare_understanding(OUTPUT / example["image"], example[group])
    with steer_understanding(
        adapter, vectors=vectors, alignment=alignment, layers=all_run_config["intervention_layers"],
        target=all_run_config["target"], operation=all_run_config["operation"],
        coefficient=all_run_config["coefficient"],
    ) as trace:
        answer = adapter.answer(embeddings, mask, max_new_tokens=all_run_config["generation"]["max_new_tokens"])
    all_rows.append(dict(
        baseline, steered=answer, steered_refusal="", steered_incoherent="", steered_irrelevant="",
        run_id=all_run_id, target=all_run_config["target"], operation=all_run_config["operation"],
        coefficient=all_run_config["coefficient"],
        intervention_layers=json.dumps(all_run_config["intervention_layers"]),
    ))
    all_traces.append(dict(sample_id=example["id"], instruction_type=group, alignment=alignment, **trace))
    print(example["id"], "|", group, "| completed")


In [ ]:
# 21. Compare baseline and steered responses.
for (example, group, _), row in zip(all_inputs, all_rows):
    show_example(OUTPUT, example, row["instruction"], row["baseline"], row["steered"])


In [ ]:
# 22. Save this run independently; refusal labels remain empty.
all_run.mkdir(parents=True, exist_ok=True)
save_json(all_run / "configuration.json", all_run_config)
save_table(all_run / "responses.json", all_rows)
save_json(all_run / "trace.json", all_traces)
print("Saved:", all_run)
